# FILE 3 — TRUSTWORTHINESS & EXPLAINABILITY WRAPPER (ResNet50 + VGG19 + Morphological Stack)

Takes the three trained branches of the SIPaKMeD pipeline and adds everything needed to argue the
system is **trustworthy**, not just accurate:

| § | What | Output |
|---|------|--------|
| 0 | Image import, unified split, morphology features, **CNN training (2-stage + TTA)**, morphology stack | trained checkpoints, probability tensors |
| A | Weighted meta-ensemble (weights tuned on **validation** only) + calibration (ECE, temperature scaling) | `fusion_*.csv`, reliability diagram |
| B | MC-Dropout uncertainty (entropy, BALD), error-detection AUROC, coverage–accuracy / selective prediction | `uncertainty_*.csv`, `coverage_accuracy.csv` |
| C | OOD gate — Dissonant-Chord co-activation detector on ResNet50 features (CIFAR-10 as OOD) | `ood_gate.json` |
| D | Grad-CAM attention split into nucleus / cytoplasm / background (uses the morphology masks) | `gradcam_attention.csv` |
| E | SHAP on the morphological stack, aggregated to clinical feature groups | `shap_group_importance.csv` |
| F | Pre-registered clinical priors → **explanation reliability** metrics MAAS · CMA · CFF · ICSC | `xai_reliability.csv` |
| G | Grounded LLM clinical report (numbers only — the model never sees the image) | `clinical_report_exhibits.json` |
| H | End-to-end inference wrapper: image → OOD gate → 3 branches → fusion → uncertainty → defer/accept → explanation | `classify()` |

**Design rule:** the OOD gate is evaluated *separately* (§C). It is **not** applied to the labelled test set when
scoring classification — all test cells are valid cytology, so a gate rejection there is a false rejection (an OOD
error), not a classification error. The gate is only applied in the deployment path (§H).

**Inputs (attach on Kaggle → *Add Input*)** — everything is auto-discovered under `/kaggle/input`; override in the config cell if needed:

* `pap-png` (CNN images) and `bmp-with-morph` (BMP + SIPaKMeD `.dat` contours, only needed if the morphology features/masks are not attached);
* `cifar-10-python` (the Kaggle dataset containing `cifar-10-batches-py/`) — the out-of-distribution set for the OOD gate;
* **CNN branches:** by default (`TRAIN_CNNS = True`) ResNet50 and VGG19 are **trained here** on the unified split — Stage 1 (frozen backbone, head only), Stage 2 (partial fine-tune), then 5-view TTA — exactly as in the backbone-benchmark notebook. Set `TRAIN_CNNS = False` to instead load existing `resnet50_final.keras` / `vgg19_final.keras` (and, optionally, their `*_{test,val}_probabilities.npz`, so fusion uses the TTA probabilities you already reported);
* *(optional)* the morphology notebook's `cell_features_enhanced.csv` + `masks/` — otherwise they are regenerated here (a few minutes);
* *(optional)* Kaggle secret `GROQ_API_KEY` for §G (a templated fallback is used otherwise).

The morphology stack is **re-trained here with the identical recipe** as the morphology notebook (same split, seed, 15-model
leaderboard, top-5 stack) because that notebook does not persist the fitted stack. The fitted bundle is saved at the end so later runs can load it.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 0 — CONFIG, IMPORTS, ARTIFACT DISCOVERY
# ════════════════════════════════════════════════════════════════════════════
import os, gc, re, json, glob, math, time, pickle, shutil, warnings
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import layers, models, applications
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                             cohen_kappa_score, roc_auc_score, classification_report)

SEED = 1337
np.random.seed(SEED); tf.random.set_seed(SEED)

# ── data ────────────────────────────────────────────────────────────────────
PNG_DIR = '/kaggle/input/datasets/sazedkarim/pap-png/png'
BMP_DIR = '/kaggle/input/datasets/sazedkarim/bmp-with-morph/BMP Data'   # only used if features/masks are missing
OUT_DIR = 'trust_xai_output'

IMAGE_SIZE  = (224, 224)
NUM_CLASSES = 5
TEST_SIZE, VAL_SIZE = 0.15, 0.15          # must equal the training notebooks — the split is re-derived, then verified
CANONICAL_CLASSES = ['Dyskeratotic', 'Koilocytotic', 'Metaplastic', 'Parabasal', 'Superficial-Intermediate']
CLASS_TO_ID = {c: i for i, c in enumerate(CANONICAL_CLASSES)}

# ── CNN branches (slug -> display name). Slug == checkpoint prefix of the benchmark notebook.
CNN_SLUGS = {'resnet50': 'ResNet50', 'vgg19': 'VGG19'}
CNN_PATHS = {'resnet50': None, 'vgg19': None}      # None = auto-discover  <slug>_final.keras  (only used when TRAIN_CNNS = False)
TRAIN_CNNS = True                                  # True: train both CNNs here (hours on Kaggle GPU); False: load existing checkpoints
CNN_INIT_WEIGHTS = 'imagenet'                      # needs Kaggle Internet ON for the first download; None = random init (smoke tests)
PROB_DIR  = None                                   # None = auto-discover  <slug>_{test,val}_probabilities.npz

# ── CNN training recipe (identical to the backbone-benchmark notebook) ─────
TRAIN_BATCH_SIZE, TTA_BATCH_SIZE, TTA_STEPS = 16, 8, 5
DETERMINISTIC_TTA = False      # False = stochastic h/v-flip views as in the benchmark; True = deterministic flips
STAGE1_EPOCHS, STAGE1_LR, STAGE1_PATIENCE = 30, 1e-3, 7
STAGE2_EPOCHS, STAGE2_LR, STAGE2_PATIENCE = 100, 1e-5, 10
RLROP_FACTOR, RLROP_PATIENCE, RLROP_MIN_LR = 0.2, 4, 1e-7

# ── morphology branch ───────────────────────────────────────────────────────
MORPH_FEATURES_CSV = None     # None = auto-discover cell_features_enhanced.csv (else regenerated from BMP_DIR)
MORPH_BUNDLE_PATH  = None     # optional pickle written by this notebook on a previous run (skips re-training)
TOP_N_MODELS       = 5        # size of the morphology stack (same as the morphology notebook)

# ── fusion ──────────────────────────────────────────────────────────────────
FIXED_WEIGHTS = None          # e.g. {'morph': .2, 'resnet50': .4, 'vgg19': .4}; None = tune on VALIDATION
WEIGHT_STEP   = 0.05

# ── trust / XAI ─────────────────────────────────────────────────────────────
MC_T            = 20          # MC-Dropout passes
MC_ON_VAL       = True        # needed to calibrate the deployment defer-threshold on validation
TARGET_ACC      = 0.99        # accuracy the automated (non-deferred) cases should reach
N_CAM           = 200         # Grad-CAM subsample of the test set
N_SHAP          = 300         # SHAP subsample of the test set
OOD_MODEL       = 'resnet50'  # backbone whose penultimate features feed the OOD gate
OOD_PERCENTILE  = 10          # threshold = this percentile of held-out ID scores  (=> ~90 % ID acceptance)
OOD_K_RANGE     = range(2, 51)
N_OOD_CIFAR     = 2000        # CIFAR-10 images: first half selects k, second half evaluates
CIFAR_DIR       = None        # folder holding data_batch_1..5 (cifar-10-batches-py); None = auto-discover, then keras download
GROQ_MODEL      = 'llama-3.3-70b-versatile'

SEARCH_ROOTS = ['/kaggle/input', '/kaggle/working', '.']
def find_files(pattern):
    hits = []
    for r in SEARCH_ROOTS:
        if os.path.exists(r):
            hits += glob.glob(os.path.join(r, '**', pattern), recursive=True)
    return sorted(set(hits))

for d in ['figures', 'reports', 'models', 'morph']:
    os.makedirs(f'{OUT_DIR}/{d}', exist_ok=True)

# ── discover CNN checkpoints / probability tensors ──────────────────────────
for slug in ([] if TRAIN_CNNS else CNN_SLUGS):
    if CNN_PATHS[slug] is None:
        hits = find_files(f'{slug}_final.keras')
        assert hits, f'{slug}_final.keras not found — attach the model dataset or set CNN_PATHS["{slug}"]'
        CNN_PATHS[slug] = hits[0]
    print(f'{slug:<9} checkpoint : {CNN_PATHS[slug]}')

def _prob_file(slug, split):
    if PROB_DIR:
        p = os.path.join(PROB_DIR, f'{slug}_{split}_probabilities.npz')
        return p if os.path.exists(p) else None
    hits = find_files(f'{slug}_{split}_probabilities.npz')
    return hits[0] if hits else None

print(f'TensorFlow {tf.__version__} | Keras {tf.keras.__version__} | GPUs {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 1 — MORPHOLOGY ARTIFACTS  (features CSV + nucleus/cytoplasm masks)
# Re-used if attached; otherwise regenerated with the SAME two-tier segmentation
# (SIPaKMeD expert contours first, Otsu fallback) as the morphology notebook.
# ════════════════════════════════════════════════════════════════════════════
import cv2
from skimage.feature import graycomatrix, graycoprops

def _canonical_name(folder_name):
    """'4_im_Superficial-Intermediate' -> 'Superficial-Intermediate'"""
    parts = folder_name.split('_im_')
    return parts[-1] if len(parts) > 1 else folder_name

def _stem(p):
    return os.path.splitext(os.path.basename(p))[0]

def _read_dat_contour(path):
    try:    arr = np.loadtxt(path, delimiter=',')
    except Exception:
        try:    arr = np.loadtxt(path)
        except Exception: return None
    arr = np.atleast_2d(arr)
    if arr.ndim != 2 or arr.shape[1] < 2 or arr.shape[0] < 3: return None
    return np.round(arr[:, :2]).astype(np.int32)

def _rasterise(contours, shape):
    mask, filled = np.zeros(shape, np.uint8), 0
    for c in contours:
        if c is None: continue
        cv2.fillPoly(mask, [c.reshape(-1, 1, 2)], 255); filled += 1
    return mask if filled else None

def segment_from_groundtruth(image_path, gray_shape):
    base = os.path.splitext(image_path)[0]
    nuc_files, cyt_files = sorted(glob.glob(base + '_nuc*.dat')), sorted(glob.glob(base + '_cyt*.dat'))
    if not nuc_files or not cyt_files: return None, None
    nuc  = _rasterise([_read_dat_contour(f) for f in nuc_files], gray_shape)
    cell = _rasterise([_read_dat_contour(f) for f in cyt_files], gray_shape)
    if nuc is None or cell is None: return None, None
    cyto = cv2.bitwise_and(cell, cv2.bitwise_not(nuc))        # whole-cell contour minus nucleus
    if np.sum(nuc == 255) < 50 or np.sum(cyto == 255) < 50: return None, None
    return nuc, cyto

def segment_cell_parts_otsu(gray):
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    op = cv2.morphologyEx(th, cv2.MORPH_OPEN, k, iterations=2)
    op = cv2.morphologyEx(op, cv2.MORPH_CLOSE, k, iterations=1)
    dist = cv2.distanceTransform(op, cv2.DIST_L2, 5)
    if dist.max() == 0: return None, None
    _, nuc = cv2.threshold(dist, 0.50 * dist.max(), 255, 0); nuc = np.uint8(nuc)
    return nuc, cv2.bitwise_and(op, cv2.bitwise_not(nuc))

def segment_cell_parts(image_path):
    bgr = cv2.imread(image_path)
    if bgr is None: return None, None, None, None, None
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    nuc, cyto = segment_from_groundtruth(image_path, gray.shape)
    if nuc is not None: return nuc, cyto, bgr, gray, 'GT'
    nuc, cyto = segment_cell_parts_otsu(gray)
    if nuc is None: return None, None, bgr, gray, None
    return nuc, cyto, bgr, gray, 'OTSU'

def _circ(a, p): return float(np.clip(4 * math.pi * a / p ** 2, 0, 1)) if p > 0 else 0.0

def shape_features_from_mask(mask, pre):
    f, area = {}, float(np.sum(mask == 255))
    f[f'{pre}_area'], f[f'{pre}_log_area'] = area, np.log1p(area)
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        for k in ['perimeter','circularity','aspect_ratio','extent','solidity','equiv_diameter','hu1','hu2','hu3','hu4']:
            f[f'{pre}_{k}'] = 0.0
        return f
    cnt = max(cnts, key=cv2.contourArea); per = cv2.arcLength(cnt, True)
    x, y, w, h = cv2.boundingRect(cnt); hull_a = cv2.contourArea(cv2.convexHull(cnt))
    eq = math.sqrt(4 * area / math.pi) if area > 0 else 0.0
    hu = cv2.HuMoments(cv2.moments(cnt)).flatten()
    hul = [-np.sign(v) * np.log10(abs(v) + 1e-10) for v in hu[:4]]
    f[f'{pre}_perimeter'] = np.log1p(per);          f[f'{pre}_circularity'] = _circ(area, per)
    f[f'{pre}_aspect_ratio'] = float(w) / h if h > 0 else 0.0
    f[f'{pre}_extent'] = area / (w * h) if w * h > 0 else 0.0
    f[f'{pre}_solidity'] = area / hull_a if hull_a > 0 else 0.0
    f[f'{pre}_equiv_diameter'] = np.log1p(eq)
    for i in range(4): f[f'{pre}_hu{i+1}'] = hul[i]
    return f

def eccentricity_from_mask(mask):
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return 0.0
    cnt = max(cnts, key=cv2.contourArea)
    if len(cnt) < 5: return 0.0
    try:
        (_, ax, _) = cv2.fitEllipse(cnt); MA, ma = max(ax), min(ax)
        return float(math.sqrt(1 - (ma / MA) ** 2)) if MA > 0 else 0.0
    except Exception: return 0.0

def intensity_features(gray, mask, pre):
    v = gray[mask == 255]
    return ({f'{pre}_intensity_mean': 0.0, f'{pre}_intensity_std': 0.0} if v.size == 0 else
            {f'{pre}_intensity_mean': float(v.mean()), f'{pre}_intensity_std': float(v.std())})

def hsv_features(bgr, mask, pre):
    hsv, out = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV), {}
    for ci, cn in enumerate('HSV'):
        v = hsv[:, :, ci][mask == 255]; out[f'{pre}_hsv_{cn}_mean'] = float(v.mean()) if v.size else 0.0
    return out

_GLCM = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation']
def glcm_features(gray, nuc):
    ys, xs = np.where(nuc == 255)
    if ys.size == 0: return {f'glcm_{p}': 0.0 for p in _GLCM}
    patch = gray[ys.min():ys.max()+1, xs.min():xs.max()+1]
    if patch.size == 0 or min(patch.shape) < 4: return {f'glcm_{p}': 0.0 for p in _GLCM}
    g = graycomatrix(patch, distances=[1, 3], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                     levels=256, symmetric=True, normed=True)
    return {f'glcm_{p}': float(graycoprops(g, p).mean()) for p in _GLCM}

def extract_all_features(nuc, cyto, bgr, gray):
    f = {}; f.update(shape_features_from_mask(nuc, 'nuc')); f.update(shape_features_from_mask(cyto, 'cyto'))
    na, ca = f['nuc_area'], f['cyto_area']; cell = na + ca
    f['nc_area_ratio']   = na / ca if ca > 0 else 0.0
    f['nuc_cell_ratio']  = na / cell if cell > 0 else 0.0
    f['cyto_cell_ratio'] = ca / cell if cell > 0 else 0.0
    f['nc_circ_ratio']     = f['nuc_circularity'] / f['cyto_circularity'] if f['cyto_circularity'] > 0 else 0.0
    f['nc_solidity_ratio'] = f['nuc_solidity'] / f['cyto_solidity'] if f['cyto_solidity'] > 0 else 0.0
    f['nuc_eccentricity'], f['cyto_eccentricity'] = eccentricity_from_mask(nuc), eccentricity_from_mask(cyto)
    f.update(intensity_features(gray, nuc, 'nuc')); f.update(intensity_features(gray, cyto, 'cyto'))
    f['intensity_contrast'] = f['cyto_intensity_mean'] - f['nuc_intensity_mean']
    f['intensity_ratio'] = f['nuc_intensity_mean'] / f['cyto_intensity_mean'] if f['cyto_intensity_mean'] > 0 else 0.0
    f.update(hsv_features(bgr, nuc, 'nuc')); f.update(hsv_features(bgr, cyto, 'cyto')); f.update(glcm_features(gray, nuc))
    return f

def run_morph_pipeline(input_dir, root):
    rows, failed = [], []
    for folder in sorted(f for f in os.listdir(input_dir) if os.path.isdir(os.path.join(input_dir, f))):
        canon = _canonical_name(folder)
        for part in ['nucleus', 'cytoplasm']: os.makedirs(f'{root}/masks/{part}/{canon}', exist_ok=True)
        imgs = sorted(glob.glob(os.path.join(input_dir, folder, '*.bmp')) + glob.glob(os.path.join(input_dir, folder, '*.png')))
        ok = Counter()
        for p in imgs:
            nuc, cyto, bgr, gray, src = segment_cell_parts(p)
            if nuc is None or np.sum(nuc == 255) < 50: failed.append((canon, p)); continue
            cv2.imwrite(f'{root}/masks/nucleus/{canon}/{_stem(p)}.png', nuc)
            cv2.imwrite(f'{root}/masks/cytoplasm/{canon}/{_stem(p)}.png', cyto)
            r = extract_all_features(nuc, cyto, bgr, gray); r.update(Image=os.path.basename(p), Class=canon, seg_source=src)
            rows.append(r); ok[src] += 1
        print(f'  {folder}: {len(imgs)} images -> GT {ok["GT"]} | OTSU {ok["OTSU"]} | failed {len(imgs)-sum(ok.values())}')
    df = pd.DataFrame(rows); df.to_csv(f'{root}/cell_features_enhanced.csv', index=False)
    return df

# ── locate or regenerate ────────────────────────────────────────────────────
csv_hits = [MORPH_FEATURES_CSV] if MORPH_FEATURES_CSV else find_files('cell_features_enhanced.csv')
csv_hits = [p for p in csv_hits if p and os.path.exists(p)]
if csv_hits and os.path.isdir(os.path.join(os.path.dirname(csv_hits[0]), 'masks', 'nucleus')):
    MORPH_CSV  = csv_hits[0]
    MASK_ROOT  = os.path.join(os.path.dirname(MORPH_CSV), 'masks')
    print(f'Re-using morphology artifacts: {MORPH_CSV}')
else:
    print(f'No attached features+masks pair found -> regenerating from {BMP_DIR}')
    run_morph_pipeline(BMP_DIR, f'{OUT_DIR}/morph')
    MORPH_CSV, MASK_ROOT = f'{OUT_DIR}/morph/cell_features_enhanced.csv', f'{OUT_DIR}/morph/masks'

df_morph = pd.read_csv(MORPH_CSV)
assert set(df_morph['Class'].unique()) == set(CANONICAL_CLASSES), 'class names differ from CANONICAL_CLASSES'
FEAT_COLS = [c for c in df_morph.columns if c not in ['Image', 'Class', 'seg_source']]
print(f'Morphology table: {df_morph.shape} | {len(FEAT_COLS)} features | provenance {df_morph["seg_source"].value_counts().to_dict() if "seg_source" in df_morph else "n/a"}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 2 — UNIFIED REGISTRY + THE SINGLE STRATIFIED SPLIT
# Identical construction to the training notebooks: registry = sorted(morph ∩ png)
# keyed on (class, stem); stratified test→val split with the same seed.
# ════════════════════════════════════════════════════════════════════════════
ds = tf.keras.utils.image_dataset_from_directory(
        PNG_DIR, label_mode='int', image_size=IMAGE_SIZE, batch_size=None, seed=SEED, shuffle=False)
png_paths = list(ds.file_paths)
all_imgs  = np.stack([im.numpy().astype(np.uint8) for im, _ in ds])          # (4049,224,224,3) uint8
del ds; gc.collect()
png_keys = [(_canonical_name(os.path.basename(os.path.dirname(p))), _stem(p)) for p in png_paths]
assert len(set(png_keys)) == len(png_keys), 'duplicate (class, stem) in PNG tree'

morph_key2row = {(r.Class, _stem(r.Image)): i for i, r in enumerate(df_morph.itertuples(index=False))}
png_key2row   = {k: i for i, k in enumerate(png_keys)}
common_keys   = sorted(set(morph_key2row) & set(png_key2row))
n_common      = len(common_keys)
print(f'morph rows {len(morph_key2row)} | png {len(png_key2row)} | unified registry {n_common}')

y_unified = np.array([CLASS_TO_ID[k[0]] for k in common_keys], dtype=np.int64)
idx_tv, idx_test = train_test_split(np.arange(n_common), test_size=TEST_SIZE, random_state=SEED, stratify=y_unified)
idx_train, idx_val = train_test_split(idx_tv, test_size=VAL_SIZE / (1.0 - TEST_SIZE), random_state=SEED,
                                      stratify=y_unified[idx_tv])
assert not (set(idx_train) & set(idx_val)) and not (set(idx_train) & set(idx_test)) and not (set(idx_val) & set(idx_test))

morph_rows = np.array([morph_key2row[k] for k in common_keys])
png_rows   = np.array([png_key2row[k]   for k in common_keys])
key_str    = lambda i: f'{common_keys[i][0]}/{common_keys[i][1]}'
test_keys  = [key_str(i) for i in idx_test]
val_keys   = [key_str(i) for i in idx_val]

y_train, y_val, y_true = y_unified[idx_train], y_unified[idx_val], y_unified[idx_test]
imgs_train = all_imgs[png_rows[idx_train]]
imgs_val   = all_imgs[png_rows[idx_val]]
test_imgs  = all_imgs[png_rows[idx_test]]
del all_imgs; gc.collect()
print(f'train {len(idx_train)} | val {len(idx_val)} | test {len(idx_test)}')

# raw morphology matrices in registry order
X_reg_raw = df_morph[FEAT_COLS].values.astype(np.float64)[morph_rows]
X_test_raw = X_reg_raw[idx_test]
col_i = {c: i for i, c in enumerate(FEAT_COLS)}

## Training the CNN branches (ResNet50, VGG19)
Reproduces the backbone-benchmark notebook on the **same unified split** (just derived above): Stage 1 trains the head with the backbone frozen (LR 1e-3);
Stage 2 unfreezes the upper backbone (VGG19 from `block3_conv1`; ResNet50 top 50 % of layers; BatchNorm always frozen) at LR 1e-5 with `ReduceLROnPlateau`;
both stages use class-weighted loss, early stopping on validation loss, and the shared augmentation block. Final predictions use **5-view test-time augmentation**
(identity, h-flip, v-flip, rot90, rot270) on validation and test; the no-TTA accuracy is reported for the ablation.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 2b — CNN TRAINING: Stage 1 (head) → Stage 2 (fine-tune) → 5-view TTA
# Skipped when TRAIN_CNNS = False (existing checkpoints are loaded in Cell 3).
# ════════════════════════════════════════════════════════════════════════════
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight

PP = {'resnet50': applications.resnet.preprocess_input, 'vgg19': applications.vgg19.preprocess_input}
CNN_ZOO = {'resnet50': dict(name='ResNet50', app=applications.ResNet50, ft_start_layer=None,          ft_fraction=0.50),
           'vgg19':    dict(name='VGG19',    app=applications.VGG19,    ft_start_layer='block3_conv1', ft_fraction=0.50)}
TRAIN_PROBS, TRAIN_LOG = {}, []

if not TRAIN_CNNS:
    print('TRAIN_CNNS = False -> existing checkpoints will be loaded in Cell 3.')
else:
    AUTOTUNE = tf.data.AUTOTUNE
    cw_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)))
    print('Class weights:', {k: round(float(v), 3) for k, v in cw_dict.items()})

    augmentation = models.Sequential([layers.RandomFlip('horizontal_and_vertical'), layers.RandomRotation(0.2),
                                      layers.RandomZoom(0.2), layers.RandomBrightness(0.1), layers.RandomContrast(0.1)],
                                     name='shared_augmentation')
    def make_ds(imgs, labels, batch, shuffle=False, augment=False):
        ds = tf.data.Dataset.from_tensor_slices((imgs, labels))                 # uint8 in memory, float32 per sample
        ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
        if shuffle: ds = ds.shuffle(len(imgs), seed=SEED)
        if augment: ds = ds.map(lambda x, y: (augmentation(tf.expand_dims(x, 0), training=True)[0], y), num_parallel_calls=AUTOTUNE)
        return ds.batch(batch).prefetch(AUTOTUNE)

    tta_fh, tta_fv = layers.RandomFlip('horizontal'), layers.RandomFlip('vertical')
    def tta_predict(model, dataset):
        """5-view TTA: identity + h-flip + v-flip + rot90 + rot270, averaged. Dataset must be unshuffled."""
        out = []
        for images, _ in dataset:
            p = model.predict(images, verbose=0)
            if DETERMINISTIC_TTA:
                p += model.predict(tf.image.flip_left_right(images), verbose=0); p += model.predict(tf.image.flip_up_down(images), verbose=0)
            else:
                p += model.predict(tta_fh(images, training=True), verbose=0);    p += model.predict(tta_fv(images, training=True), verbose=0)
            p += model.predict(tf.image.rot90(images, k=1), verbose=0); p += model.predict(tf.image.rot90(images, k=3), verbose=0)
            out.extend(p / TTA_STEPS)
        return np.asarray(out)

    def build_model(slug, weights):
        z = CNN_ZOO[slug]
        base = z['app'](include_top=False, weights=weights, input_shape=IMAGE_SIZE + (3,)); base.trainable = False
        inp = layers.Input(shape=IMAGE_SIZE + (3,))
        x = base(PP[slug](inp), training=False)
        x = layers.GlobalAveragePooling2D()(x); x = layers.BatchNormalization()(x); x = layers.Dropout(0.5)(x)
        x = layers.Dense(512, activation='relu')(x); x = layers.BatchNormalization()(x); x = layers.Dropout(0.3)(x)
        x = layers.Dense(256, activation='relu')(x); x = layers.BatchNormalization()(x)
        return models.Model(inp, layers.Dense(NUM_CLASSES, activation='softmax')(x), name=f'{slug}_clf')

    def get_backbone(model):
        for l in model.layers:
            if isinstance(l, tf.keras.Model): return l
        raise RuntimeError('no nested backbone')

    def apply_finetune(base, z):
        """Stage-2 unfreezing. BatchNorm always stays frozen (2.8k training images cannot re-estimate its moments)."""
        base.trainable = True
        if z['ft_start_layer']:
            assert z['ft_start_layer'] in [l.name for l in base.layers], z['ft_start_layer']
            on = False
            for l in base.layers:
                on = on or l.name == z['ft_start_layer']
                l.trainable = False if isinstance(l, layers.BatchNormalization) else on
        else:
            cut = int(len(base.layers) * (1.0 - z['ft_fraction']))
            for i, l in enumerate(base.layers):
                l.trainable = (i >= cut) and not isinstance(l, layers.BatchNormalization)
        return sum(l.trainable for l in base.layers), len(base.layers)

    train_ds    = make_ds(imgs_train, y_train, TRAIN_BATCH_SIZE, shuffle=True, augment=True)
    val_ds      = make_ds(imgs_val,   y_val,   TRAIN_BATCH_SIZE)
    val_tta_ds  = make_ds(imgs_val,   y_val,   TTA_BATCH_SIZE)
    test_tta_ds = make_ds(test_imgs,  y_true,  TTA_BATCH_SIZE)
    test_plain  = tf.data.Dataset.from_tensor_slices(test_imgs).map(lambda x: tf.cast(x, tf.float32)).batch(TTA_BATCH_SIZE)

    for slug, z in CNN_ZOO.items():
        print('=' * 72); print(f'TRAINING {z["name"]}'); print('=' * 72)
        tf.random.set_seed(SEED); np.random.seed(SEED)
        S1, S2 = f'{OUT_DIR}/models/{slug}_stage1.keras', f'{OUT_DIR}/models/{slug}_final.keras'
        model = build_model(slug, CNN_INIT_WEIGHTS)
        model.compile(optimizer=tf.keras.optimizers.Adam(STAGE1_LR), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        print(f'-- Stage 1 (head only): {sum(int(np.prod(v.shape)) for v in model.trainable_variables):,} trainable params')
        h1 = model.fit(train_ds, epochs=STAGE1_EPOCHS, validation_data=val_ds, class_weight=cw_dict, verbose=2,
                       callbacks=[EarlyStopping(monitor='val_loss', patience=STAGE1_PATIENCE, restore_best_weights=True),
                                  ModelCheckpoint(S1, save_best_only=True, monitor='val_loss')])
        model.load_weights(S1)
        n_tr, n_tot = apply_finetune(get_backbone(model), z)
        model.compile(optimizer=tf.keras.optimizers.Adam(STAGE2_LR), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        print(f'-- Stage 2 (fine-tune): {n_tr}/{n_tot} backbone layers unfrozen, LR {STAGE2_LR}')
        h2 = model.fit(train_ds, epochs=STAGE2_EPOCHS, initial_epoch=len(h1.epoch), validation_data=val_ds, class_weight=cw_dict, verbose=2,
                       callbacks=[EarlyStopping(monitor='val_loss', patience=STAGE2_PATIENCE, restore_best_weights=True),
                                  ModelCheckpoint(S2, save_best_only=True, monitor='val_loss'),
                                  ReduceLROnPlateau(monitor='val_loss', factor=RLROP_FACTOR, patience=RLROP_PATIENCE, min_lr=RLROP_MIN_LR)])
        # ── evaluation with TTA on the best (lowest val-loss) checkpoint ──
        ev = build_model(slug, None); ev.load_weights(S2)
        P_te, P_va = tta_predict(ev, test_tta_ds), tta_predict(ev, val_tta_ds)
        P_te_plain = ev.predict(test_plain, verbose=0)
        for nm, P, n in [('P_test', P_te, len(y_true)), ('P_val', P_va, len(y_val)), ('P_noTTA', P_te_plain, len(y_true))]:
            assert P.shape == (n, NUM_CLASSES) and np.allclose(P.sum(1), 1.0, atol=1e-4), f'{nm}: bad probabilities'
        acc = lambda P, y: accuracy_score(y, P.argmax(1))
        TRAIN_LOG.append({'Model': z['name'], 'Ep_S1': len(h1.epoch), 'Ep_S2': len(h2.epoch), 'Val_TTA': acc(P_va, y_val),
                          'Test_TTA': acc(P_te, y_true), 'Test_noTTA': acc(P_te_plain, y_true),
                          'TTA_delta_pp': (acc(P_te, y_true) - acc(P_te_plain, y_true)) * 100})
        print(f'{z["name"]}: val TTA {TRAIN_LOG[-1]["Val_TTA"]*100:.2f}% | test TTA {TRAIN_LOG[-1]["Test_TTA"]*100:.2f}% | '
              f'test no-TTA {TRAIN_LOG[-1]["Test_noTTA"]*100:.2f}% | epochs S1/S2 {len(h1.epoch)}/{len(h2.epoch)}')
        TRAIN_PROBS[(slug, 'test')], TRAIN_PROBS[(slug, 'val')] = P_te, P_va
        os.makedirs(f'{OUT_DIR}/probs', exist_ok=True)
        np.savez_compressed(f'{OUT_DIR}/probs/{slug}_test_probabilities.npz', y_true=y_true, P_test=P_te, P_test_noTTA=P_te_plain,
                            class_names=np.array(CANONICAL_CLASSES), test_keys=np.array(test_keys), model_name=np.array([z['name']]))
        np.savez_compressed(f'{OUT_DIR}/probs/{slug}_val_probabilities.npz', y_true=y_val, P_val=P_va,
                            class_names=np.array(CANONICAL_CLASSES), val_keys=np.array(val_keys), model_name=np.array([z['name']]))
        CNN_PATHS[slug] = S2
        pd.DataFrame(TRAIN_LOG).to_csv(f'{OUT_DIR}/reports/cnn_training_summary.csv', index=False)
        del model, ev, h1, h2; gc.collect()      # NB: no clear_session() — the tf.data augmentation layers live in the global graph
    print(pd.DataFrame(TRAIN_LOG).to_string(index=False, float_format=lambda v: f'{v:.4f}'))

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 3 — BRANCH 1 & 2: CNN LOADING  (explicit pre-processing, nested backbone)
#
# Keras 3 note: the `preprocess_input` step of a functional model is an *op*, not
# a Layer, so it does NOT appear in `model.layers`. Anything that replays
# `model.layers` one by one (a common Grad-CAM recipe) silently skips it and
# feeds raw 0-255 pixels to the backbone. Here the pre-processing is applied
# explicitly, and each branch is verified against `model.predict`.
# ════════════════════════════════════════════════════════════════════════════
def _apply(layer, h):
    try:    return layer(h, training=False)
    except TypeError: return layer(h)

class CNNBranch:
    """backbone (nested Model) -> head layers [GAP, BN, Dropout, Dense, BN, Dropout, Dense, BN, Dense(softmax)]"""
    def __init__(self, slug, path):
        self.slug, self.model, self.pp = slug, tf.keras.models.load_model(path), PP[slug]
        L = self.model.layers
        bi = [i for i, l in enumerate(L) if isinstance(l, tf.keras.Model)]
        assert bi, f'{slug}: no nested backbone found'
        self.base, self.head = L[bi[0]], L[bi[0] + 1:]
        drop = [i for i, l in enumerate(self.head) if isinstance(l, layers.Dropout)]
        assert drop, f'{slug}: head has no Dropout layer — MC-Dropout is undefined'
        self.pre, self.post = self.head[:drop[0]], self.head[drop[0]:]   # deterministic prefix / stochastic suffix
        assert tuple(self.model.input_shape[1:3]) == IMAGE_SIZE, f'{slug}: unexpected input size'

    def _batches(self, imgs, bs):
        for i in range(0, len(imgs), bs):
            yield tf.convert_to_tensor(imgs[i:i+bs].astype('float32'))

    def prefix_features(self, imgs, bs=32):
        """deterministic activations just before the first Dropout (GAP -> BN)"""
        out = []
        for x in self._batches(imgs, bs):
            h = self.base(self.pp(x), training=False)
            for l in self.pre: h = _apply(l, h)
            out.append(h.numpy())
        return np.concatenate(out)

    def embed(self, imgs, bs=32):
        """penultimate features (everything but the softmax layer) — input of the OOD gate"""
        out = []
        for x in self._batches(imgs, bs):
            h = self.base(self.pp(x), training=False)
            for l in self.head[:-1]: h = _apply(l, h)
            out.append(h.numpy())
        return np.concatenate(out)

    def probs_from_prefix(self, F):
        h = tf.convert_to_tensor(F)
        for l in self.post: h = _apply(l, h)
        return h.numpy()

    def mc_probs_from_prefix(self, F, T):
        """T stochastic head passes. Dropout is the ONLY stochastic op; BatchNorm stays in inference mode."""
        out = np.zeros((T, len(F), NUM_CLASSES), np.float32)
        for t in range(T):
            h = tf.convert_to_tensor(F)
            for l in self.post:
                h = tf.nn.dropout(h, rate=l.rate) if isinstance(l, layers.Dropout) else _apply(l, h)
            out[t] = h.numpy()
        return out

    def gradcam(self, img, cls):
        """L^c = ReLU(Σ_k α_k A^k),  α_k = mean ∂y^c/∂A^k   (Selvaraju et al., 2017)"""
        x = tf.convert_to_tensor(img[None].astype('float32'))
        with tf.GradientTape() as tape:
            conv = self.base(self.pp(x), training=False)
            tape.watch(conv)
            h = conv
            for l in self.head: h = _apply(l, h)
            y = h[:, cls]
        g = tape.gradient(y, conv)
        assert g is not None, 'gradient did not reach the conv feature map'
        w   = tf.reduce_mean(g, axis=(0, 1, 2))
        cam = np.maximum(tf.reduce_sum(conv[0] * w, axis=-1).numpy(), 0)
        if cam.max() > 0: cam /= cam.max()
        return cv2.resize(cam, IMAGE_SIZE)

branches = {s: CNNBranch(s, p) for s, p in CNN_PATHS.items()}

# ── verification: manual forward == model.predict ───────────────────────────
for s, b in branches.items():
    ref  = b.model.predict(test_imgs[:8].astype('float32'), verbose=0)
    mine = b.probs_from_prefix(b.prefix_features(test_imgs[:8]))
    err  = float(np.abs(ref - mine).max())
    print(f'{s:<9} manual-forward vs model.predict  max|Δ| = {err:.2e}')
    assert err < 1e-3, f'{s}: manual forward disagrees with the saved model — pre-processing mismatch'
    nz = sum(float(b.gradcam(test_imgs[j], int(ref[j].argmax())).max() > 0) for j in range(len(ref)))
    print(f'{s:<9} Grad-CAM non-zero on {int(nz)}/{len(ref)} smoke-test images')
    if nz == 0: print(f'  ⚠ {s}: every smoke-test Grad-CAM is all-zero after the ReLU (no positive class evidence). Expected only for an untrained/degenerate model — check training.')

# ── CNN probability tensors for fusion: reuse the reported TTA tensors when available ──
def _align(P, keys, ref_keys):
    pos = {str(k): i for i, k in enumerate(keys)}
    miss = [k for k in ref_keys if k not in pos]
    assert not miss, f'{len(miss)} keys missing from probability file, e.g. {miss[:3]}'
    return P[[pos[k] for k in ref_keys]]

P_cnn = {'test': {}, 'val': {}}
CNN_PROB_SOURCE = {}
for s, b in branches.items():
    for split, ref, imgs, field in [('test', test_keys, test_imgs, 'P_test'), ('val', val_keys, imgs_val, 'P_val')]:
        f = None if (s, split) in TRAIN_PROBS else _prob_file(s, split)
        if (s, split) in TRAIN_PROBS:
            P_cnn[split][s] = TRAIN_PROBS[(s, split)]
            CNN_PROB_SOURCE[f'{s}/{split}'] = 'trained here (5-view TTA)'
        elif f:
            z = np.load(f, allow_pickle=True)
            P_cnn[split][s] = _align(z[field], z[f'{split}_keys'], ref)
            CNN_PROB_SOURCE[f'{s}/{split}'] = f'npz (TTA): {f}'
        else:
            P_cnn[split][s] = b.probs_from_prefix(b.prefix_features(imgs))
            CNN_PROB_SOURCE[f'{s}/{split}'] = 'recomputed (single view, no TTA)'
    print(f'{s:<9} test acc {accuracy_score(y_true, P_cnn["test"][s].argmax(1))*100:.2f}%   '
          f'val acc {accuracy_score(y_val, P_cnn["val"][s].argmax(1))*100:.2f}%   [{CNN_PROB_SOURCE[s+"/test"][:24]}]')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 4 — BRANCH 3: MORPHOLOGICAL STACKED ENSEMBLE  (same recipe as the morphology notebook)
# imputer / IQR clip / RobustScaler are fit on TRAIN only; 15-model leaderboard
# ranked on VALIDATION accuracy; top-N base learners → StackingClassifier.
# ════════════════════════════════════════════════════════════════════════════
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier,
                              StackingClassifier, AdaBoostClassifier)
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb, lightgbm as lgb, catboost as cb

if MORPH_BUNDLE_PATH and os.path.exists(MORPH_BUNDLE_PATH):
    morph_bundle = pickle.load(open(MORPH_BUNDLE_PATH, 'rb'))
    assert morph_bundle['feature_cols'] == FEAT_COLS, 'bundle was trained on different features'
    print(f'Loaded morphology bundle: {MORPH_BUNDLE_PATH}  (stack = {morph_bundle["selected"]})')
else:
    imputer = SimpleImputer(strategy='median').fit(np.where(np.isinf(X_reg_raw[idx_train]), np.nan, X_reg_raw[idx_train]))
    _imp = imputer.transform(np.where(np.isinf(X_reg_raw[idx_train]), np.nan, X_reg_raw[idx_train]))
    q1, q3 = np.percentile(_imp, 25, axis=0), np.percentile(_imp, 75, axis=0)
    clip_lo, clip_hi = q1 - 5 * (q3 - q1), q3 + 5 * (q3 - q1)
    scaler = RobustScaler().fit(np.clip(_imp, clip_lo, clip_hi))

    def _tf(X):
        X = np.where(np.isinf(X), np.nan, X)
        return scaler.transform(np.clip(imputer.transform(X), clip_lo, clip_hi))
    Xm = _tf(X_reg_raw)

    def _ada():
        try:    return AdaBoostClassifier(random_state=SEED, algorithm='SAMME')
        except TypeError: return AdaBoostClassifier(random_state=SEED)      # newer sklearn: SAMME is the only option

    algorithms = {
        'Logistic Regression':    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED),
        'Random Forest':          RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=SEED, n_jobs=-1),
        'Extra Trees':            ExtraTreesClassifier(n_estimators=300, class_weight='balanced', random_state=SEED, n_jobs=-1),
        'Hist Gradient Boosting': HistGradientBoostingClassifier(max_iter=300, class_weight='balanced', random_state=SEED),
        'XGBoost':                xgb.XGBClassifier(eval_metric='mlogloss', random_state=SEED, n_jobs=-1),
        'LightGBM':               lgb.LGBMClassifier(random_state=SEED, n_jobs=-1, verbose=-1, class_weight='balanced'),
        'CatBoost':               cb.CatBoostClassifier(random_state=SEED, verbose=0, thread_count=-1, auto_class_weights='Balanced'),
        'SVC (RBF)':              CalibratedClassifierCV(SVC(kernel='rbf', C=10.0, class_weight='balanced', random_state=SEED), cv=3),
        'MLP (Neural Net)':       MLPClassifier(hidden_layer_sizes=(256, 128, 64), max_iter=300, early_stopping=True, random_state=SEED),
        'K-Nearest Neighbors':    KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        'Decision Tree':          DecisionTreeClassifier(class_weight='balanced', random_state=SEED),
        'Gaussian Naive Bayes':   GaussianNB(),
        'LDA':                    LinearDiscriminantAnalysis(),
        'QDA':                    QuadraticDiscriminantAnalysis(),
        'AdaBoost':               _ada(),
    }
    Xtr, Xva, Xte = Xm[idx_train], Xm[idx_val], Xm[idx_test]
    board = []
    for name, m in list(algorithms.items()):
        t0 = time.time()
        try:    m.fit(Xtr, y_train)
        except Exception as e:                                   # one failing base learner must not abort the run
            print(f'  ! {name} skipped: {type(e).__name__}: {str(e)[:90]}'); del algorithms[name]; continue
        board.append({'Model': name, 'Val Accuracy': accuracy_score(y_val, m.predict(Xva)),
                      'Test Accuracy': accuracy_score(y_true, m.predict(Xte)), 'Train Time (s)': round(time.time() - t0, 1)})
    leaderboard = pd.DataFrame(board).sort_values('Val Accuracy', ascending=False).reset_index(drop=True)
    print(leaderboard.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
    leaderboard.to_csv(f'{OUT_DIR}/reports/morph_leaderboard.csv', index=False)

    selected = leaderboard.head(TOP_N_MODELS)['Model'].tolist()
    print(f'\nStack = {selected}')
    stacker = StackingClassifier(
        estimators=[(n, algorithms[n]) for n in selected],
        final_estimator=LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=SEED),
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
        stack_method='predict_proba', passthrough=True, n_jobs=-1).fit(Xtr, y_train)
    assert list(stacker.classes_) == list(range(NUM_CLASSES))
    morph_bundle = {'model': stacker, 'imputer': imputer, 'scaler': scaler, 'clip_lo': clip_lo, 'clip_hi': clip_hi,
                    'feature_cols': FEAT_COLS, 'selected': selected}
    pickle.dump(morph_bundle, open(f'{OUT_DIR}/models/morph_stacked_ensemble.pkl', 'wb'))

def morph_transform(X):
    b = morph_bundle
    X = np.where(np.isinf(X), np.nan, X)
    return b['scaler'].transform(np.clip(b['imputer'].transform(X), b['clip_lo'], b['clip_hi']))

stacker = morph_bundle['model']
X_t, X_v, X_tr_t = morph_transform(X_test_raw), morph_transform(X_reg_raw[idx_val]), morph_transform(X_reg_raw[idx_train])
P_morph_val, P_morph = stacker.predict_proba(X_v), stacker.predict_proba(X_t)
print(f'\nMorphological stack  val {accuracy_score(y_val, P_morph_val.argmax(1))*100:.2f}%   '
      f'test {accuracy_score(y_true, P_morph.argmax(1))*100:.2f}%')

## A · Meta-ensemble and calibration
Fusion weights are searched on the **validation** split only (simplex grid); the test split is touched once.
Calibration uses one-parameter temperature scaling fitted on validation, then reported on test (ECE, 15 bins).

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 5 — WEIGHTED META-ENSEMBLE + CALIBRATION                     (§A)
#   P_fused = Σ_b w_b · P_b ,   w tuned on VALIDATION accuracy (ties: lower NLL)
# ════════════════════════════════════════════════════════════════════════════
import itertools
from scipy.optimize import minimize_scalar

BR  = ['morph'] + list(CNN_SLUGS)                       # branch order
Pv  = {'morph': P_morph_val, **P_cnn['val']}
Pt  = {'morph': P_morph,     **P_cnn['test']}

def fuse(P, w):  return sum(w[n] * P[n] for n in BR)
def nll(P, y):   return float(-np.log(P[np.arange(len(y)), y] + 1e-12).mean())

if FIXED_WEIGHTS:
    W = {n: float(FIXED_WEIGHTS[n]) for n in BR}
    assert abs(sum(W.values()) - 1) < 1e-9, 'FIXED_WEIGHTS must sum to 1'
    W_SOURCE = 'fixed (user)'
else:
    K = int(round(1 / WEIGHT_STEP)); best = None
    for combo in itertools.product(range(K + 1), repeat=len(BR) - 1):
        if sum(combo) > K: continue
        w = dict(zip(BR, [c / K for c in combo] + [(K - sum(combo)) / K]))
        P = fuse(Pv, w); key = (accuracy_score(y_val, P.argmax(1)), -nll(P, y_val))
        if best is None or key > best[0]: best = (key, w)
    W, W_SOURCE = best[1], f'grid search on validation (step {WEIGHT_STEP})'
print('Fusion weights:', {k: round(v, 3) for k, v in W.items()}, '|', W_SOURCE)

P_meta_val, P_meta = fuse(Pv, W), fuse(Pt, W)
meta_pred = P_meta.argmax(1)

# ── accuracy table with bootstrap CI ────────────────────────────────────────
def boot_ci(y, p, n=1000, seed=SEED):
    r = np.random.default_rng(seed); a = np.empty(n)
    for i in range(n):
        s = r.integers(0, len(y), len(y)); a[i] = (y[s] == p[s]).mean()
    return np.percentile(a, [2.5, 97.5])

rows = []
for nm, P in [*{f'{n}': Pt[n] for n in BR}.items(), ('Meta-ensemble', P_meta)]:
    pr = P.argmax(1); lo, hi = boot_ci(y_true, pr)
    rows.append({'Model': nm, 'Accuracy': accuracy_score(y_true, pr), 'CI_lo': lo, 'CI_hi': hi,
                 'BalAcc': balanced_accuracy_score(y_true, pr), 'MacroF1': f1_score(y_true, pr, average='macro'),
                 'Kappa': cohen_kappa_score(y_true, pr), 'NLL': nll(P, y_true)})
fusion_df = pd.DataFrame(rows)
print(fusion_df.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
fusion_df.to_csv(f'{OUT_DIR}/reports/fusion_results.csv', index=False)
print('\n', classification_report(y_true, meta_pred, target_names=CANONICAL_CLASSES, digits=4, zero_division=0))

# ── temperature scaling (validation) and ECE (test) ─────────────────────────
def temper(P, T):
    z = np.log(P + 1e-12) / T; z -= z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
def ece_score(P, y, bins=15):
    conf, pred = P.max(1), P.argmax(1); edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(e)

T_OPT = float(minimize_scalar(lambda T: nll(temper(P_meta_val, T), y_val), bounds=(0.05, 10), method='bounded').x)
P_cal, P_cal_val = temper(P_meta, T_OPT), temper(P_meta_val, T_OPT)
calib = {'T': T_OPT, 'ECE_before': ece_score(P_meta, y_true), 'ECE_after': ece_score(P_cal, y_true),
         'NLL_before': nll(P_meta, y_true), 'NLL_after': nll(P_cal, y_true),
         'mean_conf_before': float(P_meta.max(1).mean()), 'mean_conf_after': float(P_cal.max(1).mean())}
print(f'Temperature T = {T_OPT:.3f} | ECE {calib["ECE_before"]:.4f} -> {calib["ECE_after"]:.4f} | '
      f'NLL {calib["NLL_before"]:.4f} -> {calib["NLL_after"]:.4f}')

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, (ttl, P) in zip(ax, [('Before temperature scaling', P_meta), ('After', P_cal)]):
    edges = np.linspace(0, 1, 11); c, pr = P.max(1), P.argmax(1); xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (c > lo) & (c <= hi)
        if m.sum() >= 3: xs.append(c[m].mean()); ys.append((pr[m] == y_true[m]).mean())
    a.plot([0, 1], [0, 1], 'k--', lw=1); a.plot(xs, ys, 'o-'); a.set_title(f'{ttl}\nECE={ece_score(P, y_true):.4f}')
    a.set_xlabel('confidence'); a.set_ylabel('accuracy'); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/figures/calibration.png', dpi=150); plt.show()

## B · Uncertainty (MC-Dropout) and selective prediction
Dropout is the only stochastic element, and it lives in the classification head. The backbone and `GAP→BN` prefix are therefore
evaluated **once** and only the head is sampled `T` times with BatchNorm in inference mode. This is identical in distribution to
cloning the model with always-on Dropout, but about `T`× cheaper and cannot be contaminated by batch statistics.

`p̂ = mean_t P_t` · `H[p̂]` (total) · `E[H]` (aleatoric) · `BALD = H[p̂] − E[H]` (epistemic), fused across the two CNNs with the ensemble's own CNN weights.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 6 — MC-DROPOUT: ENTROPY / BALD, ERROR-DETECTION, COVERAGE–ACCURACY    (§B)
# ════════════════════════════════════════════════════════════════════════════
def entropy(p, eps=1e-12): return np.clip(-np.sum(p * np.log(p + eps), axis=-1), 0.0, None)
def uncertainty(stack):
    p_hat = stack.mean(0); H = entropy(p_hat); EH = entropy(stack).mean(0)
    return p_hat, H, EH, H - EH

cnn_w = np.array([W[s] for s in CNN_SLUGS]); cnn_w = cnn_w / cnn_w.sum() if cnn_w.sum() > 0 else np.ones(len(cnn_w)) / len(cnn_w)
CNN_W = dict(zip(CNN_SLUGS, cnn_w))

def mc_uncertainty(imgs, tag):
    out = {}
    for s, b in branches.items():
        F = b.prefix_features(imgs)
        S = b.mc_probs_from_prefix(F, MC_T)
        p_hat, H, EH, B = uncertainty(S)
        out[s] = dict(p_hat=p_hat, H=H, EH=EH, BALD=B, P_det=b.probs_from_prefix(F))
        print(f'  {tag:<4} {s:<9} mean BALD {B.mean():.4f} | mean aleatoric {EH.mean():.4f} | '
              f'|p̂_MC − p_det| {np.abs(p_hat - out[s]["P_det"]).mean():.4f}')
    out['BALD'] = sum(CNN_W[s] * out[s]['BALD'] for s in CNN_SLUGS)
    return out

print(f'MC-Dropout, T = {MC_T}')
MC_TEST = mc_uncertainty(test_imgs, 'test')
BALD, ENTROPY = MC_TEST['BALD'], entropy(P_cal)
correct = (meta_pred == y_true)

print('\n══ Uncertainty vs correctness (test) ══')
rows = []
measures = [('BALD (fused)', BALD), ('Entropy (calibrated meta)', ENTROPY), ('1 − max prob', 1 - P_cal.max(1))] + \
           [(f'BALD {s}', MC_TEST[s]['BALD']) for s in CNN_SLUGS]
for nm, u in measures:
    r = {'Measure': nm, 'Correct': u[correct].mean(), 'Incorrect': u[~correct].mean() if (~correct).any() else np.nan}
    r['Ratio'] = r['Incorrect'] / max(r['Correct'], 1e-9)
    r['AUROC_error_detection'] = roc_auc_score(~correct, u) if 0 < (~correct).sum() < len(correct) else np.nan
    rows.append(r)
unc_df = pd.DataFrame(rows)
print(unc_df.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
unc_df.to_csv(f'{OUT_DIR}/reports/uncertainty_separation.csv', index=False)
print('\n  Ratio ≈ 1 and AUROC ≈ 0.5 => the uncertainty carries no information about the model\'s own errors.')

def coverage_curve(unc, y, pred, levels=np.arange(0.10, 1.001, 0.05)):
    order, rows = np.argsort(unc), []
    for c in levels:
        k = max(1, int(round(c * len(y)))); s = order[:k]
        rows.append({'coverage': float(c), 'n': int(k), 'accuracy': accuracy_score(y[s], pred[s])})
    return pd.DataFrame(rows)

cov_bald, cov_ent = coverage_curve(BALD, y_true, meta_pred), coverage_curve(ENTROPY, y_true, meta_pred)
full_acc = accuracy_score(y_true, meta_pred)
print('\n══ Coverage–accuracy (ranked by fused BALD) ══')
print(cov_bald.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
cov_bald.to_csv(f'{OUT_DIR}/reports/coverage_accuracy.csv', index=False)
cov_ent.to_csv(f'{OUT_DIR}/reports/coverage_accuracy_entropy.csv', index=False)
hit = cov_bald[cov_bald.accuracy >= TARGET_ACC]
AUTOMATION = ({'coverage': float(hit.iloc[-1].coverage), 'accuracy': float(hit.iloc[-1].accuracy), 'n_automated': int(hit.iloc[-1].n)}
              if len(hit) else None)
print(f'\nFull-coverage accuracy {full_acc*100:.2f}% | ' +
      (f'≥{TARGET_ACC*100:.0f}% up to {AUTOMATION["coverage"]*100:.0f}% coverage '
       f'({AUTOMATION["n_automated"]} automated, {len(y_true)-AUTOMATION["n_automated"]} deferred)'
       if AUTOMATION else f'accuracy never reaches {TARGET_ACC*100:.0f}% at any coverage — report that plainly'))

# ── deployment defer-threshold, chosen on VALIDATION and then checked on test ──
DEFER_BALD, DEFER_INFO = None, {}
if MC_ON_VAL:
    MC_VAL = mc_uncertainty(imgs_val, 'val')
    cv = coverage_curve(MC_VAL['BALD'], y_val, P_meta_val.argmax(1), levels=np.arange(0.10, 1.001, 0.01))
    ok = cv[cv.accuracy >= TARGET_ACC]
    cov_sel = round(float(ok.iloc[-1].coverage), 2) if len(ok) else 0.5
    DEFER_BALD = float(np.quantile(MC_VAL['BALD'], cov_sel))
    auto = BALD <= DEFER_BALD
    DEFER_INFO = {'target_acc': TARGET_ACC, 'val_coverage_selected': cov_sel, 'threshold_BALD': DEFER_BALD,
                  'target_reachable_on_val': bool(len(ok)),
                  'test_automated_fraction': float(auto.mean()),
                  'test_accuracy_automated': float(correct[auto].mean()) if auto.any() else None,
                  'test_accuracy_deferred': float(correct[~auto].mean()) if (~auto).any() else None}
    print('\nDefer rule (BALD > threshold → send to pathologist):', json.dumps(DEFER_INFO, indent=1))

fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
ax[0].plot(cov_bald.coverage * 100, cov_bald.accuracy * 100, 'o-', label='ranked by BALD')
ax[0].plot(cov_ent.coverage * 100, cov_ent.accuracy * 100, 's--', label='ranked by entropy')
ax[0].axhline(full_acc * 100, ls=':', c='r', label=f'full = {full_acc*100:.2f}%')
ax[0].set(xlabel='Coverage (%)', ylabel='Accuracy (%)', title='Selective prediction'); ax[0].legend(); ax[0].grid(alpha=.3)
bins = np.linspace(0, max(BALD.max(), 1e-6), 40)
ax[1].hist(BALD[correct], bins=bins, alpha=.6, density=True, label='correct')
if (~correct).any(): ax[1].hist(BALD[~correct], bins=bins, alpha=.6, density=True, label='incorrect')
ax[1].set(xlabel='fused BALD (nats)', title='Epistemic uncertainty vs correctness'); ax[1].legend()
ax[2].scatter(ENTROPY, BALD, c=~correct, cmap='coolwarm', s=14, alpha=.7)
ax[2].set(xlabel='entropy of calibrated meta posterior', ylabel='fused BALD', title='Aleatoric+epistemic (red = error)')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/figures/uncertainty.png', dpi=150); plt.show()

## C · OOD gate
Reproduces the repo's *Dissonant-Chord Search*: a co-activation matrix of the top-`k` penultimate features is estimated on **in-distribution training**
cells; a sample's score is the mean co-activation of its own top-`k` features. Differences from the stand-alone OOD notebook, all in the direction of
less optimistic numbers: the matrix is fit on the *train* split only; `k` is chosen on *validation* ID vs the first half of the CIFAR-10 sample; the
threshold is the validation ID percentile; everything is reported on the *test* ID cells vs the **second, unseen** half of CIFAR-10.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 7 — OOD GATE (Dissonant-Chord co-activation)                          (§C)
# ════════════════════════════════════════════════════════════════════════════
def _topk(F, k): return np.argpartition(-F, k - 1, axis=1)[:, :k]

def dcs_fit(F, k):
    mask = np.zeros(F.shape, np.float32); np.put_along_axis(mask, _topk(F, k), 1.0, axis=1)
    co = mask.T @ mask; np.fill_diagonal(co, 0.0)
    return co / (co.sum(1, keepdims=True) + 1e-8)

def dcs_score(C, F, k):
    t = _topk(F, k)
    return C[t[:, :, None], t[:, None, :]].sum((1, 2)) / max(k * (k - 1), 1e-8)

ood_b = branches[OOD_MODEL]
print(f'Embedding train/val/test with {OOD_MODEL} (penultimate layer)...')
E_tr, E_va, E_te = ood_b.embed(imgs_train), ood_b.embed(imgs_val), ood_b.embed(test_imgs)
print('feature dim:', E_tr.shape[1])

def load_cifar_local(n):
    """CIFAR-10 python version (Kaggle 'cifar-10-python'): pickled batches, rows = 3072 uint8 in CHW order."""
    d = CIFAR_DIR or next((os.path.dirname(p) for p in find_files('data_batch_1')), None)
    if d is None: return None
    xs = []
    for i in range(1, 6):
        p = os.path.join(d, f'data_batch_{i}')
        if not os.path.exists(p): break
        with open(p, 'rb') as f: xs.append(pickle.load(f, encoding='bytes')[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1))
        if sum(len(x) for x in xs) >= n: break
    return np.concatenate(xs)[:n] if xs else None

CIFAR, CIFAR_SOURCE = None, None
try:
    x_c = load_cifar_local(N_OOD_CIFAR)
    CIFAR_SOURCE = 'cifar-10-python (local input)'
    if x_c is None:
        (x_c, _), _ = tf.keras.datasets.cifar10.load_data(); x_c = x_c[:N_OOD_CIFAR]; CIFAR_SOURCE = 'keras download'
    CIFAR = tf.image.resize(x_c, IMAGE_SIZE).numpy().astype('float32')
    E_c = ood_b.embed(CIFAR); h = len(E_c) // 2; E_c_sel, E_c_eval = E_c[:h], E_c[h:]
    print(f'CIFAR-10 OOD [{CIFAR_SOURCE}]: {len(E_c)} images (k-selection {h} / evaluation {len(E_c)-h})')
except Exception as e:
    CIFAR = None
    print('CIFAR-10 unavailable (attach the cifar-10-python dataset) -> OOD AUROC cannot be computed; using k = 10:', type(e).__name__, e)

K_BEST, sweep = 10, []
if CIFAR is not None:
    for k in OOD_K_RANGE:
        C = dcs_fit(E_tr, k)
        s_id, s_od = dcs_score(C, E_va, k), dcs_score(C, E_c_sel, k)
        sweep.append({'k': k, 'AUROC_val': roc_auc_score([1]*len(s_id) + [0]*len(s_od), np.r_[s_id, s_od])})
    sweep_df = pd.DataFrame(sweep); K_BEST = int(sweep_df.loc[sweep_df.AUROC_val.idxmax(), 'k'])
    sweep_df.to_csv(f'{OUT_DIR}/reports/ood_k_sweep.csv', index=False)

CO_MATRIX  = dcs_fit(E_tr, K_BEST)
OOD_THR    = float(np.percentile(dcs_score(CO_MATRIX, E_va, K_BEST), OOD_PERCENTILE))
s_test_id  = dcs_score(CO_MATRIX, E_te, K_BEST)
OOD_RES = {'model': OOD_MODEL, 'k': K_BEST, 'threshold': OOD_THR, 'percentile': OOD_PERCENTILE,
           'id_acceptance_test': float((s_test_id >= OOD_THR).mean())}
if CIFAR is not None:
    s_ood = dcs_score(CO_MATRIX, E_c_eval, K_BEST)
    OOD_RES.update(auroc_test=float(roc_auc_score([1]*len(s_test_id) + [0]*len(s_ood), np.r_[s_test_id, s_ood])),
                   ood_rejection=float((s_ood < OOD_THR).mean()), ood_set=f'CIFAR-10 held-out half [{CIFAR_SOURCE}]')
print('OOD gate:', json.dumps(OOD_RES, indent=1))
json.dump(OOD_RES, open(f'{OUT_DIR}/reports/ood_gate.json', 'w'), indent=2)
print('\nCaveat: CIFAR-10 is a *far*-OOD set. It shows the gate rejects natural images; it does NOT show it rejects '
      'near-OOD inputs (other stains, blurred slides, other cytology). Say so in the thesis.')

## D · Grad-CAM — where do the CNNs look?
For each sampled test cell the Grad-CAM map of the **predicted class** is split into the fraction of its mass falling on the nucleus, the cytoplasm and the
background, using the expert/Otsu masks (nearest-neighbour resized to 224×224). A high background fraction means attention is leaving the cell — the clearest sign of an unreliable explanation.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 8 — GRAD-CAM ATTENTION FRACTIONS                                      (§D)
# ════════════════════════════════════════════════════════════════════════════
def masks_for(key):
    cls, stem = key.split('/', 1)
    n = cv2.imread(f'{MASK_ROOT}/nucleus/{cls}/{stem}.png', 0)
    c = cv2.imread(f'{MASK_ROOT}/cytoplasm/{cls}/{stem}.png', 0)
    if n is None or c is None: return None, None
    r = lambda m: cv2.resize(m, IMAGE_SIZE, interpolation=cv2.INTER_NEAREST) > 127
    return r(n), r(c)

N_CAM_EFF = min(N_CAM, len(test_imgs))
cam_idx   = np.random.default_rng(SEED).choice(len(test_imgs), N_CAM_EFF, replace=False)
rows, fail_mask, cam_zero = [], 0, 0
print(f'Grad-CAM for {N_CAM_EFF} test cells × {len(branches)} CNNs...')
for j, i in enumerate(cam_idx):
    if j % 25 == 0: print(f'\r  {j}/{N_CAM_EFF}', end='', flush=True)
    nuc, cyt = masks_for(test_keys[i])
    if nuc is None: fail_mask += 1; continue
    bg, cls = ~(nuc | cyt), int(meta_pred[i])
    for s, b in branches.items():
        cam = b.gradcam(test_imgs[i], cls)
        if cam.max() <= 0: cam_zero += 1; continue            # ReLU removed every activation: attention undefined, not "zero"
        tot = cam.sum() + 1e-9
        rows.append({'idx': int(i), 'model': s, 'true': CANONICAL_CLASSES[y_true[i]], 'pred': CANONICAL_CLASSES[cls],
                     'nucleus': float(cam[nuc].sum() / tot), 'cytoplasm': float(cam[cyt].sum() / tot),
                     'background': float(cam[bg].sum() / tot),
                     'area_nucleus': float(nuc.mean()), 'area_cytoplasm': float(cyt.mean())})
print(f'\n  rows={len(rows)}  mask-failures={fail_mask}  all-zero CAMs skipped={cam_zero}')
assert rows, 'No Grad-CAM rows — check MASK_ROOT'
cam_df = pd.DataFrame(rows); cam_df.to_csv(f'{OUT_DIR}/reports/gradcam_attention.csv', index=False)

att = cam_df.groupby(['true', 'model'])[['nucleus', 'cytoplasm', 'background']].mean()
print('\n══ Mean Grad-CAM attention fractions by true class ══')
print(att.to_string(float_format=lambda v: f'{v:.4f}'))
# Area-normalised view: attention share ÷ area share (1.0 = proportional to area; >1 = attends more than its size)
lift = cam_df.assign(nuc_lift=cam_df.nucleus / cam_df.area_nucleus.clip(lower=1e-6),
                     cyt_lift=cam_df.cytoplasm / cam_df.area_cytoplasm.clip(lower=1e-6)
                     ).groupby('model')[['nuc_lift', 'cyt_lift']].median()
print('\nMedian area-normalised attention (attention share ÷ area share):'); print(lift.to_string(float_format=lambda v: f'{v:.3f}'))

## E · SHAP on the morphological stack → clinical feature groups
Exact tree-path SHAP is computed on every **tree-based** member of the fitted stack (RF/ET/DT/XGBoost/LightGBM/CatBoost). Each model's attributions are
normalised to fractions of its own total |φ| per sample before averaging, so log-odds models and probability models are comparable.
`I_g = Σ_{j∈g}|φ_j| / Σ_j|φ_j|`. If the selected stack happens to contain no tree model, a random-forest *surrogate* of the stack is explained instead and its fidelity is reported.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 9 — SHAP FEATURE-GROUP ATTRIBUTIONS                                   (§E)
# ════════════════════════════════════════════════════════════════════════════
try:    import shap
except ImportError:
    import subprocess, sys; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'shap'], check=False); import shap

FEATURE_GROUPS = {
    'N/C ratios':   [c for c in FEAT_COLS if c.startswith('nc_') or c in ('nuc_cell_ratio', 'cyto_cell_ratio')],
    'Eccentricity': [c for c in FEAT_COLS if 'eccentricity' in c],
    'Intensity':    [c for c in FEAT_COLS if 'intensity' in c],
    'Colour (HSV)': [c for c in FEAT_COLS if 'hsv' in c],
    'GLCM texture': [c for c in FEAT_COLS if c.startswith('glcm_')],
}
_claimed = {c for g in FEATURE_GROUPS.values() for c in g}
FEATURE_GROUPS['Nucleus shape']   = [c for c in FEAT_COLS if c.startswith('nuc_')  and c not in _claimed]
FEATURE_GROUPS['Cytoplasm shape'] = [c for c in FEAT_COLS if c.startswith('cyto_') and c not in _claimed]
_assigned = [c for g in FEATURE_GROUPS.values() for c in g]
assert len(_assigned) == len(set(_assigned)), 'feature groups overlap'
assert set(_assigned) == set(FEAT_COLS), f'unassigned features: {set(FEAT_COLS) ^ set(_assigned)}'
print({k: len(v) for k, v in FEATURE_GROUPS.items()})

TREE_TYPES = (RandomForestClassifier, ExtraTreesClassifier, DecisionTreeClassifier,
              xgb.XGBClassifier, lgb.LGBMClassifier, cb.CatBoostClassifier)
fitted = dict(stacker.named_estimators_)
tree_members = {n: m for n, m in fitted.items() if isinstance(m, TREE_TYPES)}
if tree_members:
    SHAP_SOURCE = f'exact TreeSHAP on stack members: {list(tree_members)}'
else:
    sur = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1).fit(X_tr_t, stacker.predict(X_tr_t))
    fid = float((sur.predict(X_t) == stacker.predict(X_t)).mean())
    tree_members = {'RF-surrogate': sur}
    SHAP_SOURCE = f'RF surrogate of the stack (test agreement with stack = {fid:.4f}) — stack has no tree member'
print('SHAP source:', SHAP_SOURCE)

N_SHAP_EFF = min(N_SHAP, len(X_t))
shap_idx = np.random.default_rng(SEED).choice(len(X_t), N_SHAP_EFF, replace=False)
Xs, F_ = X_t[shap_idx], len(FEAT_COLS)

def _to_nfc(sv, n):
    """normalise the many SHAP return layouts to (N, F, C)"""
    sv = np.stack(sv, -1) if isinstance(sv, list) else np.asarray(sv)
    if sv.ndim == 2: sv = sv[:, :, None]
    f_ax = [a for a in range(3) if sv.shape[a] in (F_, F_ + 1)][0]
    c_ax = [a for a in range(3) if a != f_ax and sv.shape[a] == NUM_CLASSES][0]
    n_ax = [a for a in range(3) if a not in (f_ax, c_ax)][0]
    sv = np.transpose(sv, (n_ax, f_ax, c_ax))
    return sv[:, :F_, :]

fracs = []
for nm, m in tree_members.items():
    sv = np.abs(_to_nfc(shap.TreeExplainer(m).shap_values(Xs, check_additivity=False)
                        if isinstance(m, (RandomForestClassifier, ExtraTreesClassifier)) else shap.TreeExplainer(m).shap_values(Xs), len(Xs)))
    fracs.append(sv / (sv.sum(1, keepdims=True) + 1e-12))          # per (sample, class) fraction over features
SHAP_FRAC = np.mean(fracs, 0)                                       # (N, F, C)
print('SHAP tensor:', SHAP_FRAC.shape)

def group_importance(vec):
    tot = vec.sum() + 1e-12
    return {g: float(vec[[col_i[c] for c in cols]].sum() / tot) for g, cols in FEATURE_GROUPS.items()}

y_sub, grp_rows = y_true[shap_idx], []
for ci, cname in enumerate(CANONICAL_CLASSES):
    m = (y_sub == ci)
    if m.sum(): grp_rows.append({'Class': cname, **group_importance(SHAP_FRAC[m][:, :, ci].mean(0))})
grp_df = pd.DataFrame(grp_rows).set_index('Class')
print('\n══ Mean SHAP group importance (fraction of |φ|, true-class output) ══')
print(grp_df.to_string(float_format=lambda v: f'{v:.4f}'))
grp_df.to_csv(f'{OUT_DIR}/reports/shap_group_importance.csv')

top_feats = []
for ci, cname in enumerate(CANONICAL_CLASSES):
    m = (y_sub == ci)
    if m.sum():
        imp = SHAP_FRAC[m][:, :, ci].mean(0)
        top_feats += [{'Class': cname, 'rank': r + 1, 'feature': FEAT_COLS[j], 'importance': float(imp[j])}
                      for r, j in enumerate(np.argsort(-imp)[:5])]
pd.DataFrame(top_feats).to_csv(f'{OUT_DIR}/reports/shap_top_features.csv', index=False)

## F · Clinical priors and explanation reliability
The priors below must be fixed **before** looking at any attribution (otherwise CFF is unfalsifiable). They are the cytology-textbook expectations for each class
and are written to Appendix B together with their citations. **Please verify each citation against your reference list before submission.**

| Metric | Question it answers |
|---|---|
| **MAAS** `1 − ½Σ_r\|I_r^SHAP − I_r^CAM\|` | Do the morphology model (SHAP) and the CNNs (Grad-CAM) agree on *nucleus vs cytoplasm*? |
| **CMA** `cos(a_res, a_vgg)` | Do two different CNN architectures attend to the same regions? |
| **CFF** `\|top3(SHAP) ∩ prior\| / 3` | Does the model use the features a cytopathologist would? (low ⇒ shortcut learning) |
| **ICSC** mean pairwise ρ of \|φ\| within a class | Is the explanation class-level or instance-specific? |

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 10 — CLINICAL PRIORS (APPENDIX B) — fixed a priori
# ════════════════════════════════════════════════════════════════════════════
CLINICAL_PRIORS = {
  'Superficial-Intermediate': {
    'groups': ['Cytoplasm shape', 'N/C ratios', 'Intensity'],
    'rationale': 'Mature squamous cells: abundant polygonal cytoplasm, small pyknotic nucleus, hence a very low N/C ratio.',
    'citation': 'Plissiti et al. (2018), IEEE ICIP, pp. 3144-3148; Nayar & Wilbur (2015), The Bethesda System, 3rd ed.'},
  'Parabasal': {
    'groups': ['N/C ratios', 'Nucleus shape', 'Cytoplasm shape'],
    'rationale': 'Immature basal-layer cells: scant cytoplasm with a proportionally large round nucleus, giving a high N/C ratio.',
    'citation': 'Plissiti et al. (2018); Nayar & Wilbur (2015).'},
  'Koilocytotic': {
    'groups': ['GLCM texture', 'Intensity', 'Nucleus shape'],
    'rationale': 'HPV cytopathic effect: perinuclear cavitation with a dense peripheral rim — a localised texture/intensity '
                 'phenomenon plus nuclear enlargement and wrinkling.',
    'citation': 'Koss & Melamed (2006), Koss Diagnostic Cytology, 5th ed.; Plissiti et al. (2018).'},
  'Dyskeratotic': {
    'groups': ['Colour (HSV)', 'Intensity', 'Nucleus shape'],
    'rationale': 'Premature keratinisation: intensely eosinophilic (orangeophilic) cytoplasm with hyperchromatic nuclei — '
                 'primarily a colour and intensity signature.',
    'citation': 'Koss & Melamed (2006); Nayar & Wilbur (2015).'},
  'Metaplastic': {
    'groups': ['Cytoplasm shape', 'Eccentricity', 'N/C ratios'],
    'rationale': 'Squamous metaplasia: intermediate maturation with dense cytoplasm and often elongated/spindled outlines '
                 'distinguishing it from parabasal cells.',
    'citation': 'Plissiti et al. (2018); Koss & Melamed (2006).'},
}
def _match(cname):
    low = cname.lower()
    for k in CLINICAL_PRIORS:
        if k.split('-')[0].lower() in low: return k
PRIOR_BY_CANON = {c: CLINICAL_PRIORS[_match(c)] for c in CANONICAL_CLASSES if _match(c)}
assert len(PRIOR_BY_CANON) == NUM_CLASSES, f'prior mapping incomplete: {list(PRIOR_BY_CANON)}'
pd.DataFrame([{'Class': c, 'Expected groups': ', '.join(v['groups']), 'Rationale': v['rationale'], 'Citation': v['citation']}
              for c, v in PRIOR_BY_CANON.items()]).to_csv(f'{OUT_DIR}/reports/appendix_B_clinical_priors.csv', index=False)
for c, v in PRIOR_BY_CANON.items(): print(f'  {c:<26} -> {v["groups"]}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 11 — EXPLANATION-RELIABILITY METRICS                                  (§F)
# MAAS region assignment is done per FEATURE (not per group), so no feature group is silently dropped:
#   nucleus  : nuc_* (shape, intensity, colour, eccentricity) and glcm_* (computed on the nucleus patch)
#   cytoplasm: cyto_*
#   both     : N/C ratios, nuc_/cyto_cell_ratio, intensity_contrast, intensity_ratio  -> split 50/50
# ════════════════════════════════════════════════════════════════════════════
def _region(c):
    if c.startswith('nc_') or c in ('nuc_cell_ratio', 'cyto_cell_ratio', 'intensity_contrast', 'intensity_ratio'): return 'both'
    if c.startswith('nuc_') or c.startswith('glcm_'): return 'nucleus'
    if c.startswith('cyto_'): return 'cytoplasm'
    return 'both'
REG = np.array([_region(c) for c in FEAT_COLS])
print({r: int((REG == r).sum()) for r in ('nucleus', 'cytoplasm', 'both')})

rel_rows = []
for ci, cname in enumerate(CANONICAL_CLASSES):
    m = (y_sub == ci)
    # MAAS
    sub = cam_df[cam_df.true == cname]
    if len(sub) and m.sum():
        cn, cc = sub['nucleus'].mean(), sub['cytoplasm'].mean(); cam_vec = np.array([cn, cc]) / (cn + cc + 1e-9)
        imp = SHAP_FRAC[m][:, :, ci].mean(0)
        sn = imp[REG == 'nucleus'].sum() + 0.5 * imp[REG == 'both'].sum()
        sc = imp[REG == 'cytoplasm'].sum() + 0.5 * imp[REG == 'both'].sum()
        shap_vec = np.array([sn, sc]) / (sn + sc + 1e-9)
        MAAS = 1.0 - 0.5 * np.abs(shap_vec - cam_vec).sum()
    else: MAAS = np.nan
    # CMA (between the two CNN architectures, over {nucleus, cytoplasm, background})
    sl = list(CNN_SLUGS)
    A = [cam_df[(cam_df.true == cname) & (cam_df.model == s)][['nucleus', 'cytoplasm', 'background']].mean().values for s in sl]
    CMA = float(A[0] @ A[1] / (np.linalg.norm(A[0]) * np.linalg.norm(A[1]) + 1e-9)) if all(np.isfinite(a).all() for a in A) else np.nan
    # CFF
    if cname in grp_df.index:
        top3 = set(grp_df.loc[cname].sort_values(ascending=False).head(3).index)
        CFF = len(top3 & set(PRIOR_BY_CANON[cname]['groups'])) / 3.0
    else: CFF = np.nan
    # ICSC
    if m.sum() >= 3:
        Ak = SHAP_FRAC[m][:, :, ci][:60]; Cm = np.corrcoef(Ak); ICSC = float(np.nanmean(Cm[np.triu_indices_from(Cm, k=1)]))
    else: ICSC = np.nan
    f1 = f1_score(y_true == ci, meta_pred == ci, zero_division=0)
    rel_rows.append({'Class': cname, 'F1': f1, 'MAAS': MAAS, 'CMA': CMA, 'CFF': CFF, 'ICSC': ICSC,
                     'XAI_Rel': np.nanmean([MAAS, CMA, CFF, ICSC])})

rel_df = pd.DataFrame(rel_rows).set_index('Class'); rel_df.loc['MEAN'] = rel_df.mean(numeric_only=True)
print('══ Classification quality vs explanation reliability ══')
print(rel_df.to_string(float_format=lambda v: f'{v:.4f}'))
rel_df.to_csv(f'{OUT_DIR}/reports/xai_reliability.csv')
gap = rel_df.loc['MEAN', 'F1'] - rel_df.loc['MEAN', 'XAI_Rel']
print(f'\nF1 − XAI-reliability gap = {gap:+.4f}')
print('  Large positive gap => classifies well but explains poorly — report it, do not bury it.')
print('  Diagnosis: low CMA => architecture-dependent attention | low CFF => possible shortcut learning | '
      'low ICSC => instance-specific rather than class-level explanation | low MAAS => CNN and morphology disagree on where the evidence is.')

## G · Grounded clinical report
The language model receives **only numbers produced by this pipeline** (never the image). A small automatic check then verifies that every decimal
number quoted in the narrative appears in the supplied context; anything else is listed as ungrounded.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 12 — LLM CLINICAL REPORTS (grounded, with automatic number check)     (§G)
# ════════════════════════════════════════════════════════════════════════════
USE_GROQ, client = True, None
try:
    from kaggle_secrets import UserSecretsClient
    try:    from groq import Groq
    except ImportError:
        import subprocess, sys; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'groq'], check=False); from groq import Groq
    client = Groq(api_key=UserSecretsClient().get_secret('GROQ_API_KEY'))
except Exception as e:
    print('Groq unavailable -> templated fallback only:', type(e).__name__, e); USE_GROQ = False

REPORT_FEATURES = ['nc_area_ratio', 'nuc_circularity', 'cyto_circularity', 'nc_solidity_ratio',
                   'glcm_contrast', 'glcm_homogeneity', 'nuc_intensity_mean', 'cyto_hsv_H_mean']
SYSTEM = ("You are a reporting cytopathologist. You are given ONLY numerical outputs from an automated cervical cytology "
          "pipeline. You cannot see the image. Never invent or round-trip measurements: quote numbers exactly as given. "
          "Use exactly these five sections: Classification Summary, Morphological Evidence, Model Agreement, "
          "Clinical Significance, Recommendation. State that the result is decision support, not a diagnosis.")

def build_context(i, ood_score=None):
    raw = X_test_raw[i]; pc = CANONICAL_CLASSES[int(meta_pred[i])]
    ctx = {
        'predicted_class': pc, 'calibrated_confidence': round(float(P_cal[i].max()), 4),
        'posterior': {c: round(float(p), 4) for c, p in zip(CANONICAL_CLASSES, P_cal[i])},
        'predictive_entropy': round(float(ENTROPY[i]), 4), 'epistemic_BALD': round(float(BALD[i]), 4),
        'defer_to_pathologist': bool(DEFER_BALD is not None and BALD[i] > DEFER_BALD),
        'branch_predictions': {'morphological': CANONICAL_CLASSES[int(P_morph[i].argmax())],
                               **{s: CANONICAL_CLASSES[int(P_cnn['test'][s][i].argmax())] for s in CNN_SLUGS}},
        'fusion_weights': {k: round(v, 3) for k, v in W.items()},
        'ood_score': None if ood_score is None else round(float(ood_score), 4), 'ood_threshold': round(OOD_THR, 4),
        'morphology': {f: round(float(raw[col_i[f]]), 4) for f in REPORT_FEATURES if f in col_i},
        'class_level_top_shap_groups': (list(grp_df.loc[pc].sort_values(ascending=False).head(3).round(4).items())
                                        if pc in grp_df.index else None),
    }
    return ctx

def template_report(ctx):
    return (f"CLASSIFICATION SUMMARY\n{ctx['predicted_class']} (calibrated confidence {ctx['calibrated_confidence']}, "
            f"entropy {ctx['predictive_entropy']}).\n\nMORPHOLOGICAL EVIDENCE\n" +
            "\n".join(f"  {k} = {v}" for k, v in ctx['morphology'].items()) +
            f"\n\nMODEL AGREEMENT\n{ctx['branch_predictions']}\n\nCLINICAL SIGNIFICANCE\nEpistemic uncertainty (BALD) "
            f"{ctx['epistemic_BALD']}. " + ("Flagged for pathologist review." if ctx['defer_to_pathologist'] else "Within the automated range.") +
            "\n\nRECOMMENDATION\nDecision support only; confirm by expert review.")

def generate_report(ctx):
    if not USE_GROQ: return template_report(ctx), 'template'
    try:
        r = client.chat.completions.create(model=GROQ_MODEL, temperature=0.2, max_tokens=700,
                                           messages=[{'role': 'system', 'content': SYSTEM},
                                                     {'role': 'user', 'content': json.dumps(ctx, indent=2)}])
        return r.choices[0].message.content, 'groq'
    except Exception as e: return template_report(ctx), f'template (error: {type(e).__name__})'

def ungrounded_numbers(text, ctx, rtol=0.01):
    """decimal numbers in the narrative that match no value in the supplied context"""
    vals = []
    def walk(o):
        if isinstance(o, dict):    [walk(v) for v in o.values()]
        elif isinstance(o, (list, tuple)): [walk(v) for v in o]
        elif isinstance(o, bool):  pass
        elif isinstance(o, (int, float)): vals.append(float(o))
    walk(ctx); vals = np.array(vals + [v * 100 for v in vals])           # percentages are allowed too
    bad = []
    for tok in re.findall(r'(?<![\w.])\d+\.\d+', text):
        x = float(tok)
        if not (np.abs(vals - x) <= np.maximum(rtol * abs(x), 5e-4)).any(): bad.append(tok)
    return bad

conf_i = int(np.argmax(P_cal.max(1))); bald_i = int(np.argmax(BALD))
wrong = np.where(~correct)[0]; err_i = int(wrong[np.argmax(BALD[wrong])]) if len(wrong) else None
exhibits = []
for tag, i in [('high_confidence', conf_i), ('high_epistemic_uncertainty', bald_i), ('highest_uncertainty_error', err_i)]:
    if i is None: continue
    ctx = build_context(i, ood_score=dcs_score(CO_MATRIX, E_te[i:i+1], K_BEST)[0]); txt, src = generate_report(ctx)
    bad = ungrounded_numbers(txt, ctx)
    exhibits.append({'exhibit': tag, 'index': int(i), 'key': test_keys[i], 'true': CANONICAL_CLASSES[y_true[i]], 'source': src,
                     'ungrounded_numbers': bad, 'context': ctx, 'report': txt})
    print('=' * 72); print(f'EXHIBIT: {tag} | {test_keys[i]} | true={CANONICAL_CLASSES[y_true[i]]} | via {src} | ungrounded numbers: {bad or "none"}')
    print('=' * 72); print(txt[:1500])
json.dump(exhibits, open(f'{OUT_DIR}/reports/clinical_report_exhibits.json', 'w'), indent=2, default=str)

## H · End-to-end inference wrapper (deployment path)
`image → OOD gate → ResNet50 / VGG19 / morphology → weighted fusion → temperature calibration → MC-Dropout BALD → accept / defer → Grad-CAM`.
The morphology branch needs the cell's feature vector (requires nucleus/cytoplasm segmentation); if it is not supplied the CNN branches are fused with
their weights renormalised and the result says so.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 13 — END-TO-END INFERENCE WRAPPER                                     (§H)
# ════════════════════════════════════════════════════════════════════════════
def classify(img_uint8, morph_feature_row=None, explain=True):
    """img_uint8: (224,224,3) uint8 RGB. morph_feature_row: raw feature vector in FEAT_COLS order (optional)."""
    x = img_uint8.astype('float32')[None]
    # 1. OOD gate
    sc = float(dcs_score(CO_MATRIX, ood_b.embed(x), K_BEST)[0])
    if sc < OOD_THR:
        return {'accepted': False, 'decision': 'REJECT_OOD', 'ood_score': sc, 'ood_threshold': OOD_THR,
                'message': 'Input rejected by the OOD gate; it does not resemble the cervical-cytology training distribution.'}
    # 2. branches
    Pb, F = {}, {}
    for s, b in branches.items():
        F[s] = b.prefix_features(x); Pb[s] = b.probs_from_prefix(F[s])[0]
    w = dict(W)
    if morph_feature_row is not None:
        Pb['morph'] = stacker.predict_proba(morph_transform(np.asarray(morph_feature_row, float)[None]))[0]
    else:
        w['morph'] = 0.0; tot = sum(w.values())
        w = {k: (v / tot if tot > 0 else (1.0 / len(CNN_SLUGS) if k != 'morph' else 0.0)) for k, v in w.items()}   # all weight on morph? -> equal CNN weights
    P = temper(sum(w[n] * Pb[n] for n in w if n in Pb)[None], T_OPT)[0]
    # 3. epistemic uncertainty on the CNN branches
    bald = sum(CNN_W[s] * uncertainty(branches[s].mc_probs_from_prefix(F[s], MC_T))[3][0] for s in CNN_SLUGS)
    pred = int(P.argmax())
    out = {'accepted': True, 'decision': 'DEFER_TO_PATHOLOGIST' if (DEFER_BALD is not None and bald > DEFER_BALD) else 'AUTOMATED',
           'predicted_class': CANONICAL_CLASSES[pred], 'confidence': float(P.max()), 'entropy': float(entropy(P)),
           'epistemic_BALD': float(bald), 'defer_threshold_BALD': DEFER_BALD, 'ood_score': sc,
           'morphology_used': morph_feature_row is not None, 'effective_weights': {k: round(v, 3) for k, v in w.items()},
           'posterior': {c: round(float(p), 4) for c, p in zip(CANONICAL_CLASSES, P)},
           'branch_predictions': {n: CANONICAL_CLASSES[int(Pb[n].argmax())] for n in Pb}}
    if explain:
        out['gradcam_maps'] = {s: branches[s].gradcam(img_uint8, pred) for s in branches}
    return out

def _show(r): print(json.dumps({k: v for k, v in r.items() if k != 'gradcam_maps'}, indent=2))
print('── in-distribution, confident ──');        _show(classify(test_imgs[conf_i], X_test_raw[conf_i], explain=False))
print('── in-distribution, most uncertain ──');   _show(classify(test_imgs[bald_i], X_test_raw[bald_i], explain=False))
print('── CNN-only (no morphology vector) ──');   _show(classify(test_imgs[conf_i], None, explain=False))
if CIFAR is not None:
    print('── out-of-distribution (CIFAR-10 image) ──'); _show(classify(CIFAR[0].astype('uint8'), None, explain=False))

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 14 — FIGURES: Grad-CAM exemplars (per class) and SHAP-group heat-map
# ════════════════════════════════════════════════════════════════════════════
picks = []
for ci, cname in enumerate(CANONICAL_CLASSES):
    cand = [i for i in cam_idx if y_true[i] == ci and meta_pred[i] == ci and masks_for(test_keys[i])[0] is not None]
    if cand: picks.append((cname, max(cand, key=lambda i: P_cal[i].max())))
sl = list(CNN_SLUGS)
assert picks, 'no correctly-classified Grad-CAM exemplar with masks found — increase N_CAM'
fig, axes = plt.subplots(len(picks), 2 + len(sl), figsize=(2.9 * (2 + len(sl)), 2.7 * len(picks)), squeeze=False)
for r, (cname, i) in enumerate(picks):
    nuc, cyt = masks_for(test_keys[i]); img = test_imgs[i]
    axes[r, 0].imshow(img); axes[r, 0].set_ylabel(cname, fontsize=9)
    axes[r, 1].imshow(img); axes[r, 1].contour(nuc, levels=[0.5], colors='yellow', linewidths=1.2)
    axes[r, 1].contour(cyt, levels=[0.5], colors='cyan', linewidths=1.0)
    for c, s in enumerate(sl, start=2):
        axes[r, c].imshow(img); axes[r, c].imshow(branches[s].gradcam(img, int(meta_pred[i])), cmap='jet', alpha=0.45)
        axes[r, c].contour(nuc, levels=[0.5], colors='w', linewidths=0.8)
        if r == 0: axes[r, c].set_title(CNN_SLUGS[s], fontsize=10)
    for a in axes[r]: a.set_xticks([]); a.set_yticks([])
axes[0, 0].set_title('Original', fontsize=10); axes[0, 1].set_title('Segmentation', fontsize=10)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/figures/gradcam_exemplars.png', dpi=200, bbox_inches='tight'); plt.show()

col_order = ['Nucleus shape', 'Cytoplasm shape', 'N/C ratios', 'Intensity', 'Colour (HSV)', 'GLCM texture', 'Eccentricity']
gd = grp_df[[c for c in col_order if c in grp_df.columns]].rename(index={'Superficial-Intermediate': 'Superficial-Int.'})
fig, ax = plt.subplots(figsize=(9, 3.6)); im = ax.imshow(gd.values, cmap='YlOrRd', aspect='auto', vmin=0, vmax=max(0.45, gd.values.max()))
ax.set_xticks(range(gd.shape[1])); ax.set_xticklabels(gd.columns, rotation=30, ha='right', fontsize=9)
ax.set_yticks(range(gd.shape[0])); ax.set_yticklabels(gd.index, fontsize=9)
for i in range(gd.shape[0]):
    for j in range(gd.shape[1]):
        v = gd.values[i, j]; ax.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=8, color='white' if v > 0.28 else 'black')
fig.colorbar(im, ax=ax, shrink=0.85).set_label('Fraction of total |SHAP|', fontsize=9)
ax.set_title('Mean SHAP group importance by class'); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/shap_groups.png', dpi=200, bbox_inches='tight'); plt.show()
print('row sums (should be 1.000):', gd.sum(axis=1).round(4).to_dict())

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 15 — CONSOLIDATED RESULTS DUMP + ARCHIVE
# ════════════════════════════════════════════════════════════════════════════
FINAL = {
  'config':   {'seed': SEED, 'mc_T': MC_T, 'weights': W, 'weights_source': W_SOURCE, 'split': [len(idx_train), len(idx_val), len(idx_test)],
               'cnn_prob_source': CNN_PROB_SOURCE, 'morph_stack': morph_bundle['selected'], 'shap_source': SHAP_SOURCE,
               'n_cam': N_CAM_EFF, 'n_shap': N_SHAP_EFF},
  'classification': fusion_df.to_dict('records'),
  'calibration':    calib,
  'uncertainty':    {'separation': unc_df.to_dict('records'), 'coverage_bald': cov_bald.to_dict('records'),
                     'automation_at_target': AUTOMATION, 'defer_rule': DEFER_INFO},
  'ood_gate':       OOD_RES,
  'gradcam':        att.reset_index().to_dict('records'),
  'shap_groups':    grp_df.reset_index().to_dict('records'),
  'xai_reliability': rel_df.reset_index().to_dict('records'), 'f1_minus_xai_gap': float(gap),
  'appendix_B_priors': PRIOR_BY_CANON,
}
json.dump(FINAL, open(f'{OUT_DIR}/reports/TRUST_XAI_ALL_RESULTS.json', 'w'), indent=2, default=str)

print('=' * 66); print('WRAPPER COMPLETE'); print('=' * 66)
m_row = fusion_df[fusion_df.Model == 'Meta-ensemble'].iloc[0]
print(f'  Meta-ensemble accuracy   : {m_row.Accuracy*100:.2f}%  (95% CI {m_row.CI_lo*100:.2f}–{m_row.CI_hi*100:.2f})')
print(f'  ECE before → after       : {calib["ECE_before"]:.4f} → {calib["ECE_after"]:.4f}  (T={T_OPT:.2f})')
print(f'  BALD error-detect AUROC  : {unc_df.iloc[0]["AUROC_error_detection"]:.3f}   ratio err/corr {unc_df.iloc[0]["Ratio"]:.2f}×')
print(f'  OOD gate (k={K_BEST})        : ID accept {OOD_RES["id_acceptance_test"]*100:.1f}%' +
      (f' | AUROC {OOD_RES["auroc_test"]:.4f} | OOD reject {OOD_RES["ood_rejection"]*100:.1f}%' if 'auroc_test' in OOD_RES else ''))
print(f'  Mean XAI reliability     : {rel_df.loc["MEAN","XAI_Rel"]:.4f}   (F1 − reliability gap {gap:+.4f})')

import shutil
from IPython.display import FileLink, display
shutil.make_archive('/kaggle/working/trust_xai_output' if os.path.exists('/kaggle/working') else 'trust_xai_output', 'zip', OUT_DIR)
z = ('/kaggle/working/' if os.path.exists('/kaggle/working') else '') + 'trust_xai_output.zip'
print(f'\n{z} — {os.path.getsize(z)/1e6:.1f} MB'); display(FileLink(z))